In [1]:
import pandas as pd

In [3]:
# 1. Load the combined dataset
file_path = '../data/raw/Combined_Employee_Task_Data.csv'
df_combined = pd.read_csv(file_path)

print(f"Dataset loaded successfully. Total Rows: {df_combined.shape[0]}, Total Columns: {df_combined.shape[1]}\n")

Dataset loaded successfully. Total Rows: 11633, Total Columns: 22



In [4]:
# 2. Generate the Data-Quality Report
missing_counts = df_combined.isnull().sum()
missing_percentages = (df_combined.isnull().sum() / len(df_combined)) * 100

# Create a DataFrame for the report
quality_report = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing': missing_counts.values,
    'Missing %': missing_percentages.values.round(2)
})

# Sort to show columns with the most missing data at the top
quality_report = quality_report.sort_values(by='Missing %', ascending=False)

# Print the report in the exact requested format
print(f"{'Column':<30} {'Missing':<10} {'Missing %'}")
print("-" * 52)
for index, row in quality_report.iterrows():
    print(f"{row['Column']:<30} {row['Missing']:<10} {row['Missing %']}%")

Column                         Missing    Missing %
----------------------------------------------------
Original_Task_Description      11565      99.42%
Deadline_Date                  10012      86.07%
Employee_Department            77         0.66%
Timesheet_Work_Logs            74         0.64%
Work_Description               1          0.01%
Timesheet_ID                   0          0.0%
Hours_Spent                    0          0.0%
Task_Name                      0          0.0%
Task_ID                        0          0.0%
Date                           0          0.0%
Employee_Name                  0          0.0%
Employee_ID                    0          0.0%
Task_Description               0          0.0%
Project_Name                   0          0.0%
Employee_Job_Position          0          0.0%
Task_Priority                  0          0.0%
Actual_Hours_Spent             0          0.0%
Estimated_Planned_Hours        0          0.0%
Timesheet_Logs_Count           0          

In [6]:
df_combined.head()

,Timesheet_ID,Date,Task_ID,Task_Name,Work_Description,Hours_Spent,Project_Name,Employee_ID,Employee_Name,Employee_Department,...,Timesheet_Work_Logs,Original_Task_Description,Task_Priority,Estimated_Planned_Hours,Actual_Hours_Spent,Timesheet_Logs_Count,Task_Stage,Created_Date,Deadline_Date,All_Collaborating_Employees
0,TS-12331,2026-09-14,TSK-1640,Odoo SH Maintain,db backup and restore in test,0.25,Hovael Project,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,Add addons t the server | Get backup from live...,NaN,Low,0.0,14.25,27,Project Preparation,2026-02-05,NaN,W M I L Wijesinghe
1,TS-12330,2026-09-14,TSK-1881,Odoo sh Maintain,db backup and restore in test,0.25,CEYLON ECO SPICES,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,add addon and test | get odoo sh live ackup an...,NaN,Low,0.0,6.50,13,Developments,2026-03-19,NaN,W M I L Wijesinghe
2,TS-12329,2026-09-14,TSK-182,Odoo.SH Maintaing,add addons,0.25,Mihiri Bakemart (Pvt)Ltd - Development,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,add all addon and build and test on the odoo s...,NaN,Low,0.0,40.75,29,Ongoing,2025-06-09,NaN,W M I L Wijesinghe
3,TS-12328,2026-09-14,TSK-2884,Development Meeting,/,0.00,Cygnus One,EMP-55,Malshi Jayanthi,Colombo Branch,...,intern development hoveal project meeting | me...,NaN,Low,0.0,9.42,13,Miscellaneous,2026-07-06,NaN,"W M I L Wijesinghe, K R V Dias, A R M S Madusa..."
4,TS-12327,2026-09-14,TSK-405,Other Tasks (Mention on description),get privillages to charith's new github accoun...,0.50,Cygnus One,EMP-11,W M I L Wijesinghe,Research and Development (R&D),...,self ssl setup on vps | Preparing Report List ...,NaN,Low,0.0,162.18,69,Miscellaneous,2025-07-14,NaN,"H.M.C.S Thilakarathna, Sadaruwan Bandara, L H ..."


In [7]:
# 1. Check for exact duplicate rows across all columns
exact_duplicates = df_combined.duplicated().sum()
print(f"Exact Duplicate Rows: {exact_duplicates}")

# 2. Check for duplicate Timesheet IDs (These should ideally be 0)
duplicate_timesheets = df_combined.duplicated(subset=['Timesheet_ID']).sum()
print(f"Duplicate Timesheet IDs: {duplicate_timesheets}")

# 3. Check Task ID distribution (Demonstrating the one-to-many relationship)
duplicate_tasks = df_combined.duplicated(subset=['Task_ID']).sum()
unique_tasks = df_combined['Task_ID'].nunique()
print(f"\nDuplicate Task IDs: {duplicate_tasks}")
print(f"Total Unique Tasks: {unique_tasks}")

# 4. Display an example of a Task with multiple daily logs
print("\n--- Example: Single Task with Multiple Timesheet Logs ---")
# Find the task that has the highest number of timesheet logs
most_logged_task = df_combined['Task_ID'].value_counts().index[0]
example_df = df_combined[df_combined['Task_ID'] == most_logged_task]

display(example_df[['Timesheet_ID', 'Date', 'Task_ID', 'Hours_Spent', 'Employee_Name']].head())

Exact Duplicate Rows: 0
Duplicate Timesheet IDs: 0

Duplicate Task IDs: 9191
Total Unique Tasks: 2442

--- Example: Single Task with Multiple Timesheet Logs ---


,Timesheet_ID,Date,Task_ID,Hours_Spent,Employee_Name
10,TS-12287,2026-09-12,TSK-183,1.50,P H M M Mihilanga
13,TS-12318,2026-09-11,TSK-183,2.50,Malshi Jayanthi
50,TS-12259,2026-09-10,TSK-183,0.50,W M I L Wijesinghe
51,TS-12258,2026-09-10,TSK-183,0.25,W M I L Wijesinghe
95,TS-12191,2026-09-09,TSK-183,2.00,Malshi Jayanthi


In [8]:
# 1. Isolate the Target Variable (y)
target_column = 'Employee_ID' 
y = df_combined[target_column]

# 2. Define all "Leaky" and Non-Predictive columns to drop from Features (X)
leaky_and_id_columns = [
    'Employee_ID',             # The Target
    'Employee_Name',           # Direct identifier of the Target
    'Timesheet_ID',            # Unique log identifier (no predictive value)
    'Hours_Spent',             # Known only after daily work is logged
    'Actual_Hours_Spent',      # Known only after task completion
    'Timesheet_Logs_Count',    # Known only after task completion
    'Task_Stage'               # Current status (not known at assignment)
]

# 3. Create the Input Features DataFrame (X)
X = df_combined.drop(columns=leaky_and_id_columns, errors='ignore')

# 4. Verify the split
print(f"Target Variable (y) Shape: {y.shape}")
print(f"Input Features (X) Shape: {X.shape}")
print("\n--- Remaining Predictive Features in X ---")
for col in X.columns:
    print(f"- {col}")

Target Variable (y) Shape: (11633,)
Input Features (X) Shape: (11633, 15)

--- Remaining Predictive Features in X ---
- Date
- Task_ID
- Task_Name
- Work_Description
- Project_Name
- Employee_Department
- Employee_Job_Position
- Task_Description
- Timesheet_Work_Logs
- Original_Task_Description
- Task_Priority
- Estimated_Planned_Hours
- Created_Date
- Deadline_Date
- All_Collaborating_Employees


In [9]:
# 1. Isolate the Target Variable (y)
target_column = 'Employee_ID' 
y = df_combined[target_column]

# 2. Comprehensive list of Leaky and Non-Predictive columns
strict_leaky_columns = [
    # Target identifiers & descriptions
    'Employee_ID',                 
    'Employee_Name',               
    'Employee_Department',         
    'Employee_Job_Position',       
    'All_Collaborating_Employees', 
    
    # Timesheet / Post-assignment data
    'Timesheet_ID',            
    'Date',                        
    'Work_Description',            
    'Timesheet_Work_Logs',         
    'Hours_Spent',                 
    'Actual_Hours_Spent',          
    'Timesheet_Logs_Count',        
    'Task_Stage'                   
]

# 3. Create the rigorously cleaned Input Features DataFrame (X)
X = df_combined.drop(columns=strict_leaky_columns, errors='ignore')

# 4. Verify the finalized split
print(f"Target Variable (y) Shape: {y.shape}")
print(f"Refined Input Features (X) Shape: {X.shape}")
print("\n--- Final Safe Predictive Features (Known at Task Creation) ---")
for col in X.columns:
    print(f"- {col}")

Target Variable (y) Shape: (11633,)
Refined Input Features (X) Shape: (11633, 9)

--- Final Safe Predictive Features (Known at Task Creation) ---
- Task_ID
- Task_Name
- Project_Name
- Task_Description
- Original_Task_Description
- Task_Priority
- Estimated_Planned_Hours
- Created_Date
- Deadline_Date


In [10]:
# 1. Print the final list of safe predictive features
print("--- Features Available Before Assignment (X) ---")
for col in X.columns:
    print(f"- {col}")

# 2. Check missing values ONLY in these safe features
missing_X = X.isnull().sum()
missing_X_percentages = (X.isnull().sum() / len(X)) * 100

quality_report_X = pd.DataFrame({
    'Column': missing_X.index,
    'Missing': missing_X.values,
    'Missing %': missing_X_percentages.values.round(2)
})

quality_report_X = quality_report_X[quality_report_X['Missing'] > 0].sort_values(by='Missing %', ascending=False)

print("\n--- Missing Values in Predictive Features ---")
if quality_report_X.empty:
    print("No missing values found in input features.")
else:
    print(f"{'Column':<30} {'Missing':<10} {'Missing %'}")
    print("-" * 52)
    for index, row in quality_report_X.iterrows():
        print(f"{row['Column']:<30} {row['Missing']:<10} {row['Missing %']}%")

--- Features Available Before Assignment (X) ---
- Task_ID
- Task_Name
- Project_Name
- Task_Description
- Original_Task_Description
- Task_Priority
- Estimated_Planned_Hours
- Created_Date
- Deadline_Date

--- Missing Values in Predictive Features ---
Column                         Missing    Missing %
----------------------------------------------------
Original_Task_Description      11565      99.42%
Deadline_Date                  10012      86.07%


In [11]:
# 1. Drop heavily missing column
df_combined = df_combined.drop(columns=['Original_Task_Description'], errors='ignore')

# 2. Derive 'Has_Deadline' feature from 'Deadline_Date' and drop original
if 'Deadline_Date' in df_combined.columns:
    df_combined['Has_Deadline'] = df_combined['Deadline_Date'].notnull().astype(int)
    df_combined = df_combined.drop(columns=['Deadline_Date'])

# 3. Impute Text features with empty strings
text_columns = ['Timesheet_Work_Logs', 'Work_Description']
for col in text_columns:
    if col in df_combined.columns:
        df_combined[col] = df_combined[col].fillna("")

# 4. Impute Categorical feature with "Unknown"
if 'Employee_Department' in df_combined.columns:
    df_combined['Employee_Department'] = df_combined['Employee_Department'].fillna("Unknown")

# 5. Final Verification
print("--- Remaining Missing Values ---")
remaining_missing = df_combined.isnull().sum()
if remaining_missing.sum() == 0:
    print("All missing values have been successfully handled.")
else:
    print(remaining_missing[remaining_missing > 0])

--- Remaining Missing Values ---
All missing values have been successfully handled.


In [12]:
# Verify all columns have 0 missing values
final_missing_check = df_combined.isnull().sum()

print(f"{'Column':<30} {'Missing Count'}")
print("-" * 45)
for index, value in final_missing_check.items():
    print(f"{index:<30} {value}")

print(f"\nTotal Missing Values in Dataset: {final_missing_check.sum()}")

Column                         Missing Count
---------------------------------------------
Timesheet_ID                   0
Date                           0
Task_ID                        0
Task_Name                      0
Work_Description               0
Hours_Spent                    0
Project_Name                   0
Employee_ID                    0
Employee_Name                  0
Employee_Department            0
Employee_Job_Position          0
Task_Description               0
Timesheet_Work_Logs            0
Task_Priority                  0
Estimated_Planned_Hours        0
Actual_Hours_Spent             0
Timesheet_Logs_Count           0
Task_Stage                     0
Created_Date                   0
All_Collaborating_Employees    0
Has_Deadline                   0

Total Missing Values in Dataset: 0


In [14]:
# 1. Check current dataset size
total_rows, total_columns = df_combined.shape
print(f"Current Dataset - Rows: {total_rows}, Columns: {total_columns}\n")

Current Dataset - Rows: 11633, Columns: 21



In [15]:
categorical_columns = [
    'Employee_Department', 
    'Employee_Job_Position', 
    'Task_Priority', 
    'Task_Stage', 
    'Project_Name'
]

# 1. Safe Baseline Standardization: Strip trailing/leading spaces and convert to Title Case
for col in categorical_columns:
    if col in df_combined.columns:
        df_combined[col] = df_combined[col].astype(str).str.strip().str.title()

# 2. Inspect Unique Values
for col in categorical_columns:
    if col in df_combined.columns:
        unique_values = sorted(df_combined[col].unique().tolist())
        print(f"--- {col} ({len(unique_values)} unique values) ---")
        for val in unique_values:
            print(f"  - '{val}'")
        print("\n")

--- Employee_Department (8 unique values) ---
  - 'Administration'
  - 'Business Solution'
  - 'Colombo Branch'
  - 'Quality Assurance (Qa)'
  - 'Research And Development (R&D)'
  - 'Sales & Marketing'
  - 'Support & Maintenance'
  - 'Unknown'


--- Employee_Job_Position (20 unique values) ---
  - 'Assistent Project Manager'
  - 'Associate Functional Consultant'
  - 'Associate Pre Sales Consultant'
  - 'Associate Software Engineer'
  - 'Financial Executive'
  - 'Functional Support Executive'
  - 'Hod - Business Solutions'
  - 'Hod - Research And Development (R&D)'
  - 'Odoo Functional Consultant'
  - 'Project Manager'
  - 'Quality Assurance - Intern'
  - 'Sales & Marketing Executive'
  - 'Senior Support Executive'
  - 'Senior Support Executive L'
  - 'Software Engineer'
  - 'Support Intern'
  - 'Team Lead - Research And Development (R&D)'
  - 'Training Odoo Functional Consultant'
  - 'Training Quality Assurance'
  - 'Training Software Engineer - Internship'


--- Task_Priority (2 uniqu